# ML v1.1 long-run - перебор конфигураций CatBoost

Этот ноутбук можно запускать, когда есть время подождать. Он не меняет датасет, а проверяет несколько более сильных конфигураций CatBoost на той же версии `client_profile_v1_0_shuffled.csv`. Цель - понять, можно ли улучшить `PR-AUC` относительно текущего `v1.1` результата (`CatBoost test PR-AUC около 0.446`) и подобрать более осмысленный порог для triage.

По умолчанию обучение идет на CPU, потому что в Colab уже возникала CUDA-ошибка CatBoost. Если GPU в конкретной среде работает стабильно, можно поменять `TASK_TYPE = "CPU"` на `TASK_TYPE = "GPU"`, но для надежного unattended-запуска лучше оставить CPU.


In [2]:
!nvidia-smi

Tue Apr  7 10:53:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              8W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
from pathlib import Path
import sys
import subprocess
import importlib.util
import warnings

warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("Google Drive mount skipped:", exc)

BASE_PATH = Path("/content/drive/MyDrive/ieee-db")
DATA_FILE = BASE_PATH / "client_profile_v1_0_shuffled.csv"

RESULTS_PATH = BASE_PATH / "ml_v1_1_longrun_results.csv"
THRESHOLDS_PATH = BASE_PATH / "ml_v1_1_longrun_thresholds.csv"
TOPK_PATH = BASE_PATH / "ml_v1_1_longrun_topk.csv"
TEST_THRESHOLDS_PATH = BASE_PATH / "ml_v1_1_longrun_test_thresholds.csv"
IMPORTANCE_PATH = BASE_PATH / "ml_v1_1_longrun_best_feature_importance.csv"
MODEL_PATH = BASE_PATH / "ml_v1_1_longrun_best_catboost.cbm"

print("Файл данных:", DATA_FILE)
print("RESULTS_PATH:", RESULTS_PATH)


Mounted at /content/drive
Файл данных: /content/drive/MyDrive/ieee-db/client_profile_v1_0_shuffled.csv
RESULTS_PATH: /content/drive/MyDrive/ieee-db/ml_v1_1_longrun_results.csv


In [4]:
if importlib.util.find_spec("catboost") is None:
    print("CatBoost не найден. Устанавливаю catboost...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "catboost"])
else:
    print("CatBoost уже установлен.")

import time
import numpy as np
import pandas as pd

from catboost import CatBoostClassifier, Pool
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 180)


CatBoost не найден. Устанавливаю catboost...


In [5]:
df = pd.read_csv(DATA_FILE)
print("Размер таблицы:", df.shape)
print("Распределение target:")
print(df["profile_fraud_label"].value_counts(dropna=False))
print("Доля fraud:", df["profile_fraud_label"].mean())


Размер таблицы: (174280, 95)
Распределение target:
profile_fraud_label
0    167785
1      6495
Name: count, dtype: int64
Доля fraud: 0.03726761533165022


In [6]:
TARGET_COL = "profile_fraud_label"
ID_COLS = ["client_id"]
LEAKY_COLS = [c for c in ["tx_count_fraud", "fraud_rate"] if c in df.columns]
HIGH_MISSING_THRESHOLD = 0.95
USE_SPARSE_IDENTITY = False

if USE_SPARSE_IDENTITY:
    high_missing_cols = []
else:
    high_missing_cols = [
        c for c in df.columns
        if c not in [TARGET_COL] + ID_COLS and df[c].isna().mean() > HIGH_MISSING_THRESHOLD
    ]

feature_drop_cols = ID_COLS + LEAKY_COLS + high_missing_cols
X = df.drop(columns=[c for c in feature_drop_cols + [TARGET_COL] if c in df.columns]).copy()
y = df[TARGET_COL].astype(int).copy()

categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_cols = [c for c in X.columns if c not in categorical_cols]

print("Удаляем из признаков:", feature_drop_cols)
print("Осталось признаков:", X.shape[1])
print("Числовых признаков:", len(numeric_cols))
print("Категориальных признаков:", len(categorical_cols))
print("Категориальные колонки:", categorical_cols)


Удаляем из признаков: ['client_id', 'id_07_mean', 'id_07_median', 'id_08_mean', 'id_08_median', 'id_21_mode', 'id_22_mode', 'id_23_mode', 'id_24_mode', 'id_25_mode', 'id_26_mode', 'id_27_mode']
Осталось признаков: 82
Числовых признаков: 66
Категориальных признаков: 16
Категориальные колонки: ['card4_mode', 'card6_mode', 'top_email_domain', 'id_12_mode', 'id_15_mode', 'id_16_mode', 'id_28_mode', 'id_29_mode', 'id_30_mode', 'id_31_mode', 'id_33_mode', 'id_34_mode', 'id_35_mode', 'id_36_mode', 'id_37_mode', 'id_38_mode']


In [7]:
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y,
    test_size=0.15,
    random_state=42,
    stratify=y,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=0.1764705882,
    random_state=42,
    stratify=y_train_val,
)

print("Train:", X_train.shape, "fraud rate:", y_train.mean())
print("Val:  ", X_val.shape, "fraud rate:", y_val.mean())
print("Test: ", X_test.shape, "fraud rate:", y_test.mean())


Train: (121996, 82) fraud rate: 0.037271713826682845
Val:   (26142, 82) fraud rate: 0.037258052176574095
Test:  (26142, 82) fraud rate: 0.037258052176574095


In [8]:
def prepare_catboost_frame(frame):
    out = frame.copy()
    for c in categorical_cols:
        out[c] = out[c].astype("object").where(out[c].notna(), "MISSING").astype(str)
    return out

cat_feature_indices = [X_train.columns.get_loc(c) for c in categorical_cols]

X_train_cb = prepare_catboost_frame(X_train)
X_val_cb = prepare_catboost_frame(X_val)
X_test_cb = prepare_catboost_frame(X_test)

train_pool = Pool(X_train_cb, y_train, cat_features=cat_feature_indices)
val_pool = Pool(X_val_cb, y_val, cat_features=cat_feature_indices)
test_pool = Pool(X_test_cb, y_test, cat_features=cat_feature_indices)
print("CatBoost pools готовы")


CatBoost pools готовы


In [9]:
def calc_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "pred_positive_rate": y_pred.mean(),
    }


def build_threshold_table(y_true, y_prob, thresholds=None):
    if thresholds is None:
        thresholds = np.round(np.arange(0.10, 0.96, 0.05), 2)
    rows = []
    for threshold in thresholds:
        y_pred = (y_prob >= threshold).astype(int)
        rows.append({
            "threshold": threshold,
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0),
            "pred_positive_rate": y_pred.mean(),
            "pred_positive_count": int(y_pred.sum()),
        })
    return pd.DataFrame(rows)


def build_topk_table(y_true, y_prob, rates=(0.01, 0.03, 0.05, 0.10, 0.15)):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    order = np.argsort(-y_prob)
    total_positive = y_true.sum()
    rows = []
    for rate in rates:
        k = max(1, int(np.ceil(len(y_true) * rate)))
        selected = order[:k]
        positives_found = int(y_true[selected].sum())
        rows.append({
            "top_rate": rate,
            "top_k": k,
            "precision_at_k": positives_found / k,
            "recall_at_k": positives_found / total_positive if total_positive else 0.0,
            "fraud_found": positives_found,
            "total_fraud": int(total_positive),
        })
    return pd.DataFrame(rows)


## Набор long-run конфигураций

Это не огромный grid search, а разумный sweep из нескольких CatBoost-настроек. `early_stopping_rounds` остановит неудачные конфигурации раньше. Если времени мало, уменьшите `MAX_CONFIGS`; если оставляете ноутбук надолго, можно оставить все конфигурации.


In [10]:
TASK_TYPE = "CPU"
MAX_CONFIGS = None  # Можно поставить 3 для быстрого прогона, None - прогнать все.

catboost_configs = [
    {"name": "cb_depth6_lr004_l2_3",  "depth": 6, "learning_rate": 0.04, "l2_leaf_reg": 3,  "iterations": 3000},
    {"name": "cb_depth5_lr004_l2_3",  "depth": 5, "learning_rate": 0.04, "l2_leaf_reg": 3,  "iterations": 3000},
    {"name": "cb_depth6_lr003_l2_6",  "depth": 6, "learning_rate": 0.03, "l2_leaf_reg": 6,  "iterations": 3500},
    {"name": "cb_depth7_lr003_l2_8",  "depth": 7, "learning_rate": 0.03, "l2_leaf_reg": 8,  "iterations": 3500},
    {"name": "cb_depth4_lr006_l2_3",  "depth": 4, "learning_rate": 0.06, "l2_leaf_reg": 3,  "iterations": 2500},
    {"name": "cb_depth6_lr005_l2_10", "depth": 6, "learning_rate": 0.05, "l2_leaf_reg": 10, "iterations": 2500},
]

if MAX_CONFIGS is not None:
    catboost_configs = catboost_configs[:MAX_CONFIGS]

print("Будет обучено конфигураций:", len(catboost_configs))
for cfg in catboost_configs:
    print(cfg)


Будет обучено конфигураций: 6
{'name': 'cb_depth6_lr004_l2_3', 'depth': 6, 'learning_rate': 0.04, 'l2_leaf_reg': 3, 'iterations': 3000}
{'name': 'cb_depth5_lr004_l2_3', 'depth': 5, 'learning_rate': 0.04, 'l2_leaf_reg': 3, 'iterations': 3000}
{'name': 'cb_depth6_lr003_l2_6', 'depth': 6, 'learning_rate': 0.03, 'l2_leaf_reg': 6, 'iterations': 3500}
{'name': 'cb_depth7_lr003_l2_8', 'depth': 7, 'learning_rate': 0.03, 'l2_leaf_reg': 8, 'iterations': 3500}
{'name': 'cb_depth4_lr006_l2_3', 'depth': 4, 'learning_rate': 0.06, 'l2_leaf_reg': 3, 'iterations': 2500}
{'name': 'cb_depth6_lr005_l2_10', 'depth': 6, 'learning_rate': 0.05, 'l2_leaf_reg': 10, 'iterations': 2500}


In [11]:
results = []
fitted_models = {}

for cfg in catboost_configs:
    name = cfg["name"]
    print("\n" + "=" * 100)
    print("Обучаю:", name, cfg)
    start = time.time()

    model = CatBoostClassifier(
        iterations=cfg["iterations"],
        learning_rate=cfg["learning_rate"],
        depth=cfg["depth"],
        l2_leaf_reg=cfg["l2_leaf_reg"],
        loss_function="Logloss",
        eval_metric="PRAUC",
        auto_class_weights="Balanced",
        random_seed=42,
        task_type=TASK_TYPE,
        thread_count=-1,
        allow_writing_files=False,
        early_stopping_rounds=150,
        verbose=200,
    )

    model.fit(train_pool, eval_set=val_pool, use_best_model=True)
    elapsed_sec = time.time() - start
    fitted_models[name] = model

    for split_name, pool, y_part in [
        ("val", val_pool, y_val),
        ("test", test_pool, y_test),
    ]:
        y_prob = model.predict_proba(pool)[:, 1]
        row = calc_metrics(y_part, y_prob, threshold=0.5)
        row.update({
            "model": name,
            "split": split_name,
            "best_iteration": model.get_best_iteration(),
            "elapsed_sec": elapsed_sec,
            "depth": cfg["depth"],
            "learning_rate": cfg["learning_rate"],
            "l2_leaf_reg": cfg["l2_leaf_reg"],
            "iterations_requested": cfg["iterations"],
        })
        results.append(row)

    interim_df = pd.DataFrame(results)
    interim_df.to_csv(RESULTS_PATH, index=False)
    print("Промежуточные результаты сохранены в:", RESULTS_PATH)
    display(interim_df.sort_values(["split", "pr_auc"], ascending=[True, False]).head(20))

results_df = pd.DataFrame(results)
results_df.to_csv(RESULTS_PATH, index=False)
print("Финальная таблица long-run метрик сохранена в:", RESULTS_PATH)
display(results_df.sort_values(["split", "pr_auc"], ascending=[True, False]))



Обучаю: cb_depth6_lr004_l2_3 {'name': 'cb_depth6_lr004_l2_3', 'depth': 6, 'learning_rate': 0.04, 'l2_leaf_reg': 3, 'iterations': 3000}
0:	learn: 0.8176134	test: 0.8075705	best: 0.8075705 (0)	total: 495ms	remaining: 24m 45s
200:	learn: 0.8960238	test: 0.8829811	best: 0.8829811 (200)	total: 1m 23s	remaining: 19m 26s
400:	learn: 0.9144930	test: 0.8961948	best: 0.8961948 (400)	total: 2m 49s	remaining: 18m 21s
600:	learn: 0.9273425	test: 0.9016554	best: 0.9016554 (600)	total: 4m 17s	remaining: 17m 7s
800:	learn: 0.9368437	test: 0.9032255	best: 0.9032368 (793)	total: 5m 45s	remaining: 15m 48s
1000:	learn: 0.9437796	test: 0.9045008	best: 0.9045561 (994)	total: 7m 14s	remaining: 14m 27s
1200:	learn: 0.9494485	test: 0.9054221	best: 0.9054221 (1200)	total: 8m 41s	remaining: 13m
1400:	learn: 0.9539795	test: 0.9057274	best: 0.9057585 (1374)	total: 10m 8s	remaining: 11m 34s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 0.9058768317
bestIteration = 1447

Shrink model to first 1

,roc_auc,pr_auc,precision,recall,f1,pred_positive_rate,model,split,best_iteration,elapsed_sec,depth,learning_rate,l2_leaf_reg,iterations_requested
1,0.893293,0.465946,0.220751,0.718686,0.337756,0.121299,cb_depth6_lr004_l2_3,test,1447,695.70469,6,0.04,3,3000
0,0.894706,0.464846,0.220759,0.740246,0.340094,0.124933,cb_depth6_lr004_l2_3,val,1447,695.70469,6,0.04,3,3000



Обучаю: cb_depth5_lr004_l2_3 {'name': 'cb_depth5_lr004_l2_3', 'depth': 5, 'learning_rate': 0.04, 'l2_leaf_reg': 3, 'iterations': 3000}
0:	learn: 0.8259309	test: 0.8200138	best: 0.8200138 (0)	total: 624ms	remaining: 31m 10s
200:	learn: 0.8916891	test: 0.8815863	best: 0.8815863 (200)	total: 1m 11s	remaining: 16m 38s
400:	learn: 0.9069271	test: 0.8936482	best: 0.8936482 (400)	total: 2m 21s	remaining: 15m 19s
600:	learn: 0.9178276	test: 0.8998370	best: 0.8998370 (600)	total: 3m 34s	remaining: 14m 15s
800:	learn: 0.9254009	test: 0.9016729	best: 0.9018450 (775)	total: 4m 47s	remaining: 13m 8s
1000:	learn: 0.9312300	test: 0.9029789	best: 0.9029789 (1000)	total: 6m	remaining: 11m 59s
1200:	learn: 0.9362701	test: 0.9036248	best: 0.9036780 (1151)	total: 7m 14s	remaining: 10m 50s
1400:	learn: 0.9406104	test: 0.9044918	best: 0.9045011 (1396)	total: 8m 26s	remaining: 9m 38s
1600:	learn: 0.9445714	test: 0.9047276	best: 0.9048073 (1595)	total: 9m 39s	remaining: 8m 26s
1800:	learn: 0.9477857	test: 0.

,roc_auc,pr_auc,precision,recall,f1,pred_positive_rate,model,split,best_iteration,elapsed_sec,depth,learning_rate,l2_leaf_reg,iterations_requested
1,0.893293,0.465946,0.220751,0.718686,0.337756,0.121299,cb_depth6_lr004_l2_3,test,1447,695.704690,6,0.04,3,3000
3,0.890599,0.461970,0.213793,0.732033,0.330935,0.127572,cb_depth5_lr004_l2_3,test,1942,760.178719,5,0.04,3,3000
0,0.894706,0.464846,0.220759,0.740246,0.340094,0.124933,cb_depth6_lr004_l2_3,val,1447,695.704690,6,0.04,3,3000
2,0.894549,0.463704,0.213052,0.747433,0.331587,0.130709,cb_depth5_lr004_l2_3,val,1942,760.178719,5,0.04,3,3000



Обучаю: cb_depth6_lr003_l2_6 {'name': 'cb_depth6_lr003_l2_6', 'depth': 6, 'learning_rate': 0.03, 'l2_leaf_reg': 6, 'iterations': 3500}
0:	learn: 0.8176109	test: 0.8075705	best: 0.8075705 (0)	total: 433ms	remaining: 25m 16s
200:	learn: 0.8924004	test: 0.8805479	best: 0.8805479 (200)	total: 1m 28s	remaining: 24m 19s
400:	learn: 0.9039411	test: 0.8895459	best: 0.8895459 (400)	total: 2m 49s	remaining: 21m 52s
600:	learn: 0.9166257	test: 0.8981648	best: 0.8981648 (600)	total: 4m 17s	remaining: 20m 43s
800:	learn: 0.9255131	test: 0.9012942	best: 0.9014667 (783)	total: 5m 47s	remaining: 19m 31s
1000:	learn: 0.9325833	test: 0.9033966	best: 0.9034207 (988)	total: 7m 15s	remaining: 18m 6s
1200:	learn: 0.9382434	test: 0.9044824	best: 0.9044972 (1199)	total: 8m 42s	remaining: 16m 41s
1400:	learn: 0.9431751	test: 0.9054635	best: 0.9054635 (1400)	total: 10m 11s	remaining: 15m 16s
1600:	learn: 0.9469302	test: 0.9061181	best: 0.9061205 (1595)	total: 11m 38s	remaining: 13m 48s
1800:	learn: 0.9501752	t

...

,roc_auc,pr_auc,precision,recall,f1,pred_positive_rate,model,split,best_iteration,elapsed_sec,depth,learning_rate,l2_leaf_reg,iterations_requested
7,0.890259,0.470827,0.236245,0.705339,0.353941,0.111239,cb_depth7_lr003_l2_8,test,1889,1063.308996,7,0.03,8,3500
5,0.892253,0.466699,0.221594,0.724846,0.339423,0.121873,cb_depth6_lr003_l2_6,test,2132,997.815472,6,0.03,6,3500
1,0.893293,0.465946,0.220751,0.718686,0.337756,0.121299,cb_depth6_lr004_l2_3,test,1447,695.704690,6,0.04,3,3000
3,0.890599,0.461970,0.213793,0.732033,0.330935,0.127572,cb_depth5_lr004_l2_3,test,1942,760.178719,5,0.04,3,3000
9,0.890165,0.461108,0.202980,0.741273,0.318693,0.136065,cb_depth4_lr006_l2_3,test,1670,523.804048,4,0.06,3,2500
6,0.896731,0.473077,0.238526,0.731006,0.359687,0.114184,cb_depth7_lr003_l2_8,val,1889,1063.308996,7,0.03,8,3500
4,0.896191,0.472584,0.221714,0.746407,0.341876,0.125430,cb_depth6_lr003_l2_6,val,2132,997.815472,6,0.03,6,3500
0,0.894706,0.464846,0.220759,0.740246,0.340094,0.124933,cb_depth6_lr004_l2_3,val,1447,695.704690,6,0.04,3,3000
8,0.895911,0.464596,0.199837,0.756674,0.316173,0.141076,cb_depth4_lr006_l2_3,val,1670,523.804048,4,0.06,3,2500
2,0.894549,0.463704,0.213052,0.747433,0.331587,0.130709,cb_depth5_lr004_l2_3,val,1942,760.178719,5,0.04,3,3000



Обучаю: cb_depth6_lr005_l2_10 {'name': 'cb_depth6_lr005_l2_10', 'depth': 6, 'learning_rate': 0.05, 'l2_leaf_reg': 10, 'iterations': 2500}
0:	learn: 0.8176082	test: 0.8075599	best: 0.8075599 (0)	total: 425ms	remaining: 17m 42s
200:	learn: 0.9003921	test: 0.8879160	best: 0.8879160 (200)	total: 1m 24s	remaining: 16m
400:	learn: 0.9194102	test: 0.8993017	best: 0.8993017 (400)	total: 2m 51s	remaining: 14m 56s
600:	learn: 0.9311863	test: 0.9036063	best: 0.9036063 (600)	total: 4m 19s	remaining: 13m 39s
800:	learn: 0.9401252	test: 0.9057842	best: 0.9057976 (799)	total: 5m 46s	remaining: 12m 15s
1000:	learn: 0.9467956	test: 0.9067957	best: 0.9069095 (956)	total: 7m 14s	remaining: 10m 51s
1200:	learn: 0.9517010	test: 0.9073788	best: 0.9076414 (1136)	total: 8m 43s	remaining: 9m 25s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 0.9076414302
bestIteration = 1136

Shrink model to first 1137 iterations.
Промежуточные результаты сохранены в: /content/drive/MyDrive/ieee-db/ml_v1_1

,roc_auc,pr_auc,precision,recall,f1,pred_positive_rate,model,split,best_iteration,elapsed_sec,depth,learning_rate,l2_leaf_reg,iterations_requested
7,0.890259,0.470827,0.236245,0.705339,0.353941,0.111239,cb_depth7_lr003_l2_8,test,1889,1063.308996,7,0.03,8,3500
5,0.892253,0.466699,0.221594,0.724846,0.339423,0.121873,cb_depth6_lr003_l2_6,test,2132,997.815472,6,0.03,6,3500
1,0.893293,0.465946,0.220751,0.718686,0.337756,0.121299,cb_depth6_lr004_l2_3,test,1447,695.704690,6,0.04,3,3000
11,0.894502,0.465320,0.214200,0.731006,0.331317,0.127152,cb_depth6_lr005_l2_10,test,1136,561.561131,6,0.05,10,2500
3,0.890599,0.461970,0.213793,0.732033,0.330935,0.127572,cb_depth5_lr004_l2_3,test,1942,760.178719,5,0.04,3,3000
9,0.890165,0.461108,0.202980,0.741273,0.318693,0.136065,cb_depth4_lr006_l2_3,test,1670,523.804048,4,0.06,3,2500
6,0.896731,0.473077,0.238526,0.731006,0.359687,0.114184,cb_depth7_lr003_l2_8,val,1889,1063.308996,7,0.03,8,3500
4,0.896191,0.472584,0.221714,0.746407,0.341876,0.125430,cb_depth6_lr003_l2_6,val,2132,997.815472,6,0.03,6,3500
10,0.898025,0.466749,0.215502,0.753593,0.335160,0.130288,cb_depth6_lr005_l2_10,val,1136,561.561131,6,0.05,10,2500
0,0.894706,0.464846,0.220759,0.740246,0.340094,0.124933,cb_depth6_lr004_l2_3,val,1447,695.704690,6,0.04,3,3000


Финальная таблица long-run метрик сохранена в: /content/drive/MyDrive/ieee-db/ml_v1_1_longrun_results.csv


,roc_auc,pr_auc,precision,recall,f1,pred_positive_rate,model,split,best_iteration,elapsed_sec,depth,learning_rate,l2_leaf_reg,iterations_requested
7,0.890259,0.470827,0.236245,0.705339,0.353941,0.111239,cb_depth7_lr003_l2_8,test,1889,1063.308996,7,0.03,8,3500
5,0.892253,0.466699,0.221594,0.724846,0.339423,0.121873,cb_depth6_lr003_l2_6,test,2132,997.815472,6,0.03,6,3500
1,0.893293,0.465946,0.220751,0.718686,0.337756,0.121299,cb_depth6_lr004_l2_3,test,1447,695.704690,6,0.04,3,3000
11,0.894502,0.465320,0.214200,0.731006,0.331317,0.127152,cb_depth6_lr005_l2_10,test,1136,561.561131,6,0.05,10,2500
3,0.890599,0.461970,0.213793,0.732033,0.330935,0.127572,cb_depth5_lr004_l2_3,test,1942,760.178719,5,0.04,3,3000
9,0.890165,0.461108,0.202980,0.741273,0.318693,0.136065,cb_depth4_lr006_l2_3,test,1670,523.804048,4,0.06,3,2500
6,0.896731,0.473077,0.238526,0.731006,0.359687,0.114184,cb_depth7_lr003_l2_8,val,1889,1063.308996,7,0.03,8,3500
4,0.896191,0.472584,0.221714,0.746407,0.341876,0.125430,cb_depth6_lr003_l2_6,val,2132,997.815472,6,0.03,6,3500
10,0.898025,0.466749,0.215502,0.753593,0.335160,0.130288,cb_depth6_lr005_l2_10,val,1136,561.561131,6,0.05,10,2500
0,0.894706,0.464846,0.220759,0.740246,0.340094,0.124933,cb_depth6_lr004_l2_3,val,1447,695.704690,6,0.04,3,3000


In [12]:
best_model_name = (
    results_df[results_df["split"] == "val"]
    .sort_values("pr_auc", ascending=False)
    .iloc[0]["model"]
)
best_model = fitted_models[best_model_name]
print("Лучшая long-run модель по val PR-AUC:", best_model_name)

val_prob = best_model.predict_proba(val_pool)[:, 1]
test_prob = best_model.predict_proba(test_pool)[:, 1]

thresholds_df = build_threshold_table(y_val, val_prob)
thresholds_df["model"] = best_model_name
thresholds_df.to_csv(THRESHOLDS_PATH, index=False)
print("Таблица порогов на val сохранена в:", THRESHOLDS_PATH)
display(thresholds_df.sort_values("f1", ascending=False))

best_threshold = thresholds_df.sort_values("f1", ascending=False).iloc[0]["threshold"]
print("Лучший threshold по val F1:", best_threshold)

selected_thresholds = sorted(set([0.70, 0.75, 0.80, 0.85, 0.90, float(best_threshold)]))
test_thresholds_df = build_threshold_table(y_test, test_prob, thresholds=selected_thresholds)
test_thresholds_df["model"] = best_model_name
test_thresholds_df.to_csv(TEST_THRESHOLDS_PATH, index=False)
print("Таблица выбранных порогов на test сохранена в:", TEST_THRESHOLDS_PATH)
display(test_thresholds_df)

topk_df = build_topk_table(y_test, test_prob)
topk_df["model"] = best_model_name
topk_df.to_csv(TOPK_PATH, index=False)
print("Top-k таблица на test сохранена в:", TOPK_PATH)
display(topk_df)

print("Classification report на test при threshold=0.5")
print(classification_report(y_test, (test_prob >= 0.5).astype(int), digits=4))
print("Classification report на test при лучшем val threshold")
print(classification_report(y_test, (test_prob >= best_threshold).astype(int), digits=4))


Лучшая long-run модель по val PR-AUC: cb_depth7_lr003_l2_8
Таблица порогов на val сохранена в: /content/drive/MyDrive/ieee-db/ml_v1_1_longrun_thresholds.csv


,threshold,precision,recall,f1,pred_positive_rate,pred_positive_count,model
15,0.85,0.535055,0.446612,0.486849,0.031099,813,cb_depth7_lr003_l2_8
14,0.80,0.463768,0.492813,0.477850,0.039591,1035,cb_depth7_lr003_l2_8
13,0.75,0.419536,0.537988,0.471435,0.047778,1249,cb_depth7_lr003_l2_8
16,0.90,0.596463,0.380903,0.464912,0.023793,622,cb_depth7_lr003_l2_8
12,0.70,0.372446,0.580082,0.453633,0.058029,1517,cb_depth7_lr003_l2_8
11,0.65,0.329121,0.614990,0.428776,0.069620,1820,cb_depth7_lr003_l2_8
10,0.60,0.291686,0.651951,0.403047,0.083276,2177,cb_depth7_lr003_l2_8
17,0.95,0.702479,0.261807,0.381451,0.013886,363,cb_depth7_lr003_l2_8
9,0.55,0.263220,0.689938,0.381060,0.097659,2553,cb_depth7_lr003_l2_8
8,0.50,0.238526,0.731006,0.359687,0.114184,2985,cb_depth7_lr003_l2_8


Лучший threshold по val F1: 0.85
Таблица выбранных порогов на test сохранена в: /content/drive/MyDrive/ieee-db/ml_v1_1_longrun_test_thresholds.csv


,threshold,precision,recall,f1,pred_positive_rate,pred_positive_count,model
0,0.70,0.385366,0.567762,0.459112,0.054893,1435,cb_depth7_lr003_l2_8
1,0.75,0.443697,0.542094,0.487985,0.045521,1190,cb_depth7_lr003_l2_8
2,0.80,0.495807,0.485626,0.490664,0.036493,954,cb_depth7_lr003_l2_8
3,0.85,0.563686,0.427105,0.485981,0.028230,738,cb_depth7_lr003_l2_8
4,0.90,0.644809,0.363450,0.464872,0.021001,549,cb_depth7_lr003_l2_8


Top-k таблица на test сохранена в: /content/drive/MyDrive/ieee-db/ml_v1_1_longrun_topk.csv


,top_rate,top_k,precision_at_k,recall_at_k,fraud_found,total_fraud,model
0,0.01,262,0.770992,0.207392,202,974,cb_depth7_lr003_l2_8
1,0.03,785,0.552866,0.445585,434,974,cb_depth7_lr003_l2_8
2,0.05,1308,0.415902,0.558522,544,974,cb_depth7_lr003_l2_8
3,0.10,2615,0.256214,0.687885,670,974,cb_depth7_lr003_l2_8
4,0.15,3922,0.188679,0.759754,740,974,cb_depth7_lr003_l2_8


Classification report на test при threshold=0.5
              precision    recall  f1-score   support

           0     0.9876    0.9118    0.9482     25168
           1     0.2362    0.7053    0.3539       974

    accuracy                         0.9041     26142
   macro avg     0.6119    0.8085    0.6511     26142
weighted avg     0.9597    0.9041    0.9260     26142

Classification report на test при лучшем val threshold
              precision    recall  f1-score   support

           0     0.9780    0.9872    0.9826     25168
           1     0.5637    0.4271    0.4860       974

    accuracy                         0.9663     26142
   macro avg     0.7709    0.7072    0.7343     26142
weighted avg     0.9626    0.9663    0.9641     26142



In [13]:
importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": best_model.get_feature_importance(),
}).sort_values("importance", ascending=False)

importance_df.to_csv(IMPORTANCE_PATH, index=False)
best_model.save_model(str(MODEL_PATH))

print("Важности признаков сохранены в:", IMPORTANCE_PATH)
print("Лучшая модель сохранена в:", MODEL_PATH)
display(importance_df.head(50))


Важности признаков сохранены в: /content/drive/MyDrive/ieee-db/ml_v1_1_longrun_best_feature_importance.csv
Лучшая модель сохранена в: /content/drive/MyDrive/ieee-db/ml_v1_1_longrun_best_catboost.cbm


,feature,importance
1,tx_amount_sum,6.914786
14,odd_amount_share,5.954778
27,top_email_domain,5.752557
3,tx_amount_median_proxy,5.739129
2,tx_amount_mean,5.390728
9,tx_freq_per_day,4.095193
11,avg_inter_tx_seconds,3.851138
4,tx_amount_std,3.849362
12,short_turnover_share,3.759813
70,id_20_mode,3.352136


## Что смотреть после long-run

Главное сравнение - `PR-AUC` на `val` и `test` относительно текущего результата `v1.1`, где CatBoost дал test `PR-AUC` около 0.446. Если long-run улучшает `PR-AUC` и при этом test не проваливается относительно validation, конфигурацию можно считать более сильным кандидатом. Затем смотрим не порог `0.5`, а таблицу выбранных порогов на test и top-k: для банковской задачи часто важнее качество верхней части списка риска, чем формальная бинарная классификация при дефолтном пороге.
